# JSON → Excel: полное разворачивание без ручного указания полей

Как это работает:
- Обычные поля (строки, числа, булевы) становятся колонками.
- Вложенные объекты (dict) "расплющиваются" с префиксом: {"a": {"b": 1}} -> колонка "a.b".
- Списки объектов (например, "screens": [...]) порождают ОТДЕЛЬНУЮ СТРОКУ на каждый элемент, при этом все поля родительских уровней автоматически копируются в каждую такую строку.
- Списки простых значений (строк/чисел, не объектов) сохраняются как одна строка через запятую.
- Колонки с одинаковым именем на разных уровнях вложенности не путаются - у каждой свой префикс пути (например, "dashboards.id" и "dashboards.screens.widget.id" - разные колонки).

Просто выполните ячейки по порядку. В последней ячейке укажите путь к своему JSON-файлу.

In [ ]:
import json
from pathlib import Path

import pandas as pd


def flatten_record(obj: dict, prefix: str = "") -> dict:
    """Собирает скалярные поля текущего уровня (без разворачивания списков объектов)."""
    flat = {}
    for k, v in obj.items():
        key = f"{prefix}{k}" if prefix else k
        if isinstance(v, dict):
            flat.update(flatten_record(v, prefix=f"{key}."))
        elif isinstance(v, list):
            if v and isinstance(v[0], dict):
                continue  # это подтаблица - развернётся отдельно в explode()
            flat[key] = ", ".join(str(x) for x in v) if v else None
        else:
            flat[key] = v
    return flat


def find_list_of_dicts_fields(obj: dict):
    return [k for k, v in obj.items() if isinstance(v, list) and v and isinstance(v[0], dict)]


def explode(obj: dict, prefix: str = "", parent_context: dict = None, rows: list = None) -> list:
    """Рекурсивно разворачивает вложенный JSON в список плоских строк-словарей."""
    if rows is None:
        rows = []
    parent_context = parent_context or {}

    own_scalars = flatten_record(obj, prefix=prefix)
    context = {**parent_context, **own_scalars}

    list_fields = find_list_of_dicts_fields(obj)
    if not list_fields:
        rows.append(context)
        return rows

    for field in list_fields:
        child_prefix = f"{prefix}{field}." if prefix else f"{field}."
        for item in obj[field]:
            if isinstance(item, dict):
                explode(item, prefix=child_prefix, parent_context=context, rows=rows)

    return rows


def explode_root(data) -> list:
    if isinstance(data, list):
        rows = []
        for item in data:
            if isinstance(item, dict):
                rows.extend(explode(item))
        return rows
    if isinstance(data, dict):
        return explode(data)
    return []

In [ ]:
# Укажите путь к вашему JSON файлу:
JSON_PATH = Path("dashboards.json")
OUTPUT_PATH = JSON_PATH.with_suffix(".xlsx")

print(f"Читаю: {JSON_PATH.resolve()}")
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

rows = explode_root(data)
print(f"Строк: {len(rows)}")

df = pd.DataFrame(rows)
print(f"Колонок: {len(df.columns)}")
print("Колонки:", list(df.columns))

df.to_excel(OUTPUT_PATH, index=False)
print(f"Сохранено: {OUTPUT_PATH.resolve()}")

df.head(20)